In [1]:
#| default_exp perf.quantization

In [2]:
#| export
import numpy as np
import time
from typing import Tuple, Dict, List, Optional
import warnings

# Import dependencies from other modules
from tinytorch.core.tensor import Tensor
from tinytorch.core.layers import Linear, Sequential
from tinytorch.core.activations import ReLU

# Constants for INT8 quantization
INT8_MIN_VALUE = -128
INT8_MAX_VALUE = 127
INT8_RANGE = 256  # Number of possible INT8 values (from -128 to 127 inclusive)
EPSILON = 1e-8  # Small value for numerical stability (constant tensor detection)

# Constants for memory calculations
BYTES_PER_FLOAT32 = 4  # Standard float32 size in bytes
BYTES_PER_INT8 = 1  # INT8 size in bytes
MB_TO_BYTES = 1024 * 1024  # Megabytes to bytes conversion

if __name__ == "__main__":
    print("Quantization module imports complete")

Quantization module imports complete


In [3]:
def explore_motivation_profiling():
    """Profile model memory usage to discover the quantization problem."""
    from tinytorch.perf.profiling import Profiler

    profiler = Profiler()

    # Create models of increasing size
    print("Profiling Memory Usage (FP32 Precision):\n")
    print("   Parameters   |  FP32 Memory  |  Device Fit?")
    print("   -------------|---------------|---------------")

    model_configs = [
        (256, 256, "Tiny"),
        (512, 512, "Small"),
        (1024, 1024, "Medium"),
        (2048, 2048, "Large"),
    ]

    for in_feat, out_feat, name in model_configs:
        model = Linear(in_feat, out_feat)
        input_data = Tensor(np.random.randn(1, in_feat))

        # Profile the model
        profile = profiler.profile_forward_pass(model, input_data)

        params = profile['parameters']
        memory_fp32_mb = params * BYTES_PER_FLOAT32 / MB_TO_BYTES
        memory_fp32_gb = memory_fp32_mb / 1000

        # Check if it fits on different devices
        fits_mobile = "Y" if memory_fp32_mb < 100 else "N"
        fits_edge = "Y" if memory_fp32_mb < 10 else "N"

        print(f"   {params:>10,}  |  {memory_fp32_mb:7.1f} MB  |  Mobile:{fits_mobile} Edge:{fits_edge}")

    print("\nKey Observations:")
    print("   Every parameter uses 4 bytes (32 bits) in FP32")
    print("   Larger models quickly exceed mobile device memory (~100MB limit)")
    print("   Edge devices have even tighter constraints (~10MB)")
    print("   Memory grows linearly with parameter count")

    print("\nThe Problem:")
    print("   Do we really need 32-bit precision for inference?")
    print("   FP32: Can represent 2^32 = 4.3 billion unique values")
    print("   Neural networks are naturally robust to noise")
    print("   Most weights are in range [-3, 3] after training")

    print("\nThe Solution:")
    print("   Quantize to INT8 (8-bit integers):")
    print("   FP32 -> INT8: 32 bits -> 8 bits (4x compression!)")
    print("   Memory: 100MB -> 25MB (now fits on mobile!)")
    print("   Speed: INT8 operations are 2-4x faster on hardware")
    print("   Accuracy: Minimal loss (<1% typically) with proper calibration\n")

if __name__ == "__main__":
    explore_motivation_profiling()

Profiling Memory Usage (FP32 Precision):

   Parameters   |  FP32 Memory  |  Device Fit?
   -------------|---------------|---------------
       65,792  |      0.3 MB  |  Mobile:Y Edge:Y
      262,656  |      1.0 MB  |  Mobile:Y Edge:Y
    1,049,600  |      4.0 MB  |  Mobile:Y Edge:Y
    4,196,352  |     16.0 MB  |  Mobile:Y Edge:N

Key Observations:
   Every parameter uses 4 bytes (32 bits) in FP32
   Larger models quickly exceed mobile device memory (~100MB limit)
   Edge devices have even tighter constraints (~10MB)
   Memory grows linearly with parameter count

The Problem:
   Do we really need 32-bit precision for inference?
   FP32: Can represent 2^32 = 4.3 billion unique values
   Neural networks are naturally robust to noise
   Most weights are in range [-3, 3] after training

The Solution:
   Quantize to INT8 (8-bit integers):
   FP32 -> INT8: 32 bits -> 8 bits (4x compression!)
   Memory: 100MB -> 25MB (now fits on mobile!)
   Speed: INT8 operations are 2-4x faster on hardwar

In [ ]:
#| export
def quantize_int8(tensor: Tensor) -> Tuple[Tensor, float, int]:
    """
    Quantize FP32 tensor to INT8 using symmetric quantization.

    TODO: Implement INT8 quantization with scale and zero_point calculation

    APPROACH:
    1. Find min/max values in tensor data
    2. Calculate scale: (max_val - min_val) / 255 (INT8 range: -128 to 127)
    3. Calculate zero_point: offset that maps min_val to INT8_MIN (-128)
       Formula: zero_point = round(INT8_MIN - min_val / scale)
    4. Apply quantization formula: round(value / scale + zero_point)
    5. Clamp to INT8 range [-128, 127]

    Args:
        tensor: Input FP32 tensor to quantize

    Returns:
        q_tensor: Quantized INT8 tensor
        scale: Scaling factor (float)
        zero_point: Zero point offset (int)

    EXAMPLE:
    >>> tensor = Tensor([[-1.0, 0.0, 2.0], [0.5, 1.5, -0.5]])
    >>> q_tensor, scale, zero_point = quantize_int8(tensor)
    >>> print(f"Scale: {scale:.4f}, Zero point: {zero_point}")
    Scale: 0.0118, Zero point: -43

    HINTS:
    - Use np.round() for quantization
    - Clamp with np.clip(values, -128, 127)
    - Handle edge case where min_val == max_val (set scale=1.0)
    """
    ### BEGIN SOLUTION
    data = tensor.data

    # step 1: find dynamic range
    min_val = float(np.min(data))
    max_val = float(np.max(data))

    # step 2: handle edge case (constant tensor)
    if abs(max_val - min_val) < EPSILON:
        scale = 1.0
        zero_point = 0
        quantized_data = np.zeros_like(data, dtype=np.int8)
        return Tensor(quantized_data), scale, zero_point
    
    # step 3: calculate scale and zero_point for standard quantization
    # map [min_val, max_val] to []
    scale = (max_val - min_val) / (INT8_RANGE - 1)
    zero_point = int(np.round(INT8_MIN_VALUE - min_val / scale))

    # clamp zero_point to valid INT8 range
    zero_point = int(np.clip(zero_point, INT8_MIN_VALUE, INT8_MAX_VALUE))

    # step 4: apply quantization formula: q = (x / scale) + zero_point
    quantized_data = np.round(data / scale + zero_point)

    # step 5: clamp to INT8 range and convert to int8
    quantized_data = np.clip(quantized_data, INT8_MIN_VALUE, INT8_MAX_VALUE).astype(np.int8)

    return Tensor(quantized_data), scale, zero_point
    ### END SOLUTION